In [1]:
import pandas as pd
import numpy as np

In [3]:
# Read the 6 CSV files into a single DataFrame and save it as a CSV file
df = pd.concat([pd.read_csv(f"modis_202{i}_Uzbekistan.csv") for i in range(0, 6)], ignore_index=True)
df.to_csv("combined_data.csv", index=False)

In [7]:
df.columns

Index(['latitude', 'longitude', 'brightness', 'scan', 'track', 'acq_date',
       'acq_time', 'satellite', 'instrument', 'confidence', 'version',
       'bright_t31', 'frp', 'daynight', 'type'],
      dtype='object')

In [8]:
# Add year to as a new column
df['year'] = df['acq_date'].str[:4].astype(int)

In [12]:
import geopandas as gpd

fires = pd.read_csv("combined_data.csv", dtype={"acq_time": "string"})
fires["acq_time"] = fires["acq_time"].str.zfill(4)

regions = (
    gpd.read_file("uzbekistan_adm1.geojson")
    .to_crs("EPSG:4326")[["shapeName", "shapeID", "geometry"]]
    .rename(columns={"shapeName": "region_name", "shapeID": "region_id"})
)

points = gpd.GeoDataFrame(
    fires,
    geometry=gpd.points_from_xy(fires["longitude"], fires["latitude"]),
    crs="EPSG:4326",
)

joined = points.sjoin(regions, how="left", predicate="within")

if joined.index.has_duplicates:
    raise ValueError("Some detections matched multiple regions. Check the boundary layer.")

result = pd.DataFrame(joined[fires.columns.tolist() + ["region_name", "region_id"]])
print(f"Rows: {len(result):,} | Unmatched: {result['region_name'].isna().sum():,}")
print(result["region_name"].value_counts(dropna=False).to_string())
result.to_csv("fires_with_regions.csv", index=False)

Rows: 18,032 | Unmatched: 320
region_name
Republic of Karakalpakstan    5021
Tashkent Region               4390
Surxondaryo Region            2363
Bukhara Region                1686
Andijan Region                 775
Sirdaryo Region                581
Samarqand Region               563
Namangan Region                547
Jizzakh Region                 540
Qashqadaryo Region             384
NaN                            320
Xorazm Region                  278
Tashkent                       220
Fergana Region                 201
Navoiy Region                  163


In [16]:
df[result['region_name'].isna()].to_csv("unmatched_fires.csv", index=False)

In [19]:
result = joined[fires.columns.tolist() + ["region_name", "region_id"]].copy()
result["region_match"] = result["region_name"].notna().map(
    {True: "Within region", False: "Unassigned"}
)

unmatched = points.loc[result.index[result["region_name"].isna()]]
max_distance_m = 10000

if not unmatched.empty:
    nearest = unmatched.to_crs("EPSG:32642").sjoin_nearest(
        regions[["region_name", "region_id", "geometry"]].to_crs("EPSG:32642"),
        how="inner",
        max_distance=max_distance_m,
        distance_col="region_distance_m",
    )

    # An exact tie can return multiple rows for the same detection.
    unique = nearest.loc[~nearest.index.duplicated(keep=False)]

    for column in ["region_name", "region_id"]:
        result.loc[unique.index, column] = unique[column]

    result.loc[unique.index, "region_match"] = "Nearest within 10 km"
    result["region_distance_m"] = unique["region_distance_m"]

print(result["region_match"].value_counts())
result.to_csv("fires_with_regions.csv", index=False)

region_match
Within region           17712
Nearest within 10 km      320
Name: count, dtype: int64


In [22]:
result["region_name"].value_counts()

region_name
Republic of Karakalpakstan    5197
Tashkent Region               4420
Surxondaryo Region            2376
Bukhara Region                1693
Andijan Region                 791
Sirdaryo Region                606
Jizzakh Region                 568
Samarqand Region               563
Namangan Region                550
Qashqadaryo Region             385
Xorazm Region                  294
Tashkent                       220
Fergana Region                 206
Navoiy Region                  163
Name: count, dtype: int64